# Mapa de selección de sitio

Dada una coordenada de referencia, dibuja la batimetría, las isóbatas elegidas y la malla de nodos de WAVERYS (Copernicus Marine) con el recuadro de cada celda, y arma la tabla de nodos candidatos.

**Antes de empezar**

1. `uv sync` en la raíz del repositorio y elegir el kernel de `.venv`.
2. `uv run copernicusmarine login` (una sola vez).
3. Indicar la carpeta de datos en `DIRECTORIO_DATOS` o con la variable de entorno `WAVE_DATA_DIR`.

Los insumos se descargan una sola vez a `raw/` con su hash; las salidas van a `interim/F01-sitio-y-fuentes/mapa-sitio/<nombre>/`.

## 1. Parámetros

In [ ]:
NOMBRE = "ejemplo-marcona"      # minúsculas: nombra la carpeta de salida
LATITUD = -15.40                 # grados, negativa al sur
LONGITUD = -75.30                # grados, negativa al oeste
ISOBATAS = [200, 100, 50]        # m; la primera es la principal
ISOBATA_PRINCIPAL = None         # None: la primera de ISOBATAS
SEMIANCHO_GRADOS = 1.0           # recuadro de ±semiancho alrededor de la referencia
N_CANDIDATOS = 10
DIRECTORIO_DATOS = None          # None: variable de entorno WAVE_DATA_DIR

BATIMETRIA = {"fuente": "gmrt", "resolucion_gmrt": "med"}
# BATIMETRIA = {"fuente": "archivo", "archivo": "C:/ruta/gebco_recorte.nc"}
WAVERYS = {"fuente": "copernicus", "fecha": "2020-07-15T12:00:00"}

## 2. Generar el mapa

In [ ]:
%matplotlib inline
from peruvian_wave_analysis.sitio import ConfigMapa, generar_mapa

cfg = ConfigMapa(
    nombre=NOMBRE, latitud=LATITUD, longitud=LONGITUD,
    isobatas=ISOBATAS, isobata_principal=ISOBATA_PRINCIPAL,
    semiancho_grados=SEMIANCHO_GRADOS, n_candidatos=N_CANDIDATOS,
    directorio_datos=DIRECTORIO_DATOS, batimetria=BATIMETRIA, waverys=WAVERYS,
)
res = generar_mapa(cfg)
res.figura

## 3. Candidatos

Nodos que cumplen los criterios básicos (nodo válido, celda sin tierra, ocho vecinas válidas), ordenados por distancia a la isóbata principal. El número coincide con el del mapa.

In [ ]:
principal = f"dist_iso_{cfg.isobata_principal:g}m_km"
columnas = ["candidato", "lat", "lon", "prof_nodo_m", "prof_min_celda_m", "prof_media_celda_m",
            principal, "dist_costa_km", "dist_referencia_km"]
res.tabla[res.tabla["candidato"].notna()][columnas].head(cfg.n_candidatos).round(2)

Nodos válidos que **no** cumplen (celda con tierra o vecinas sin dato): pueden estar más cerca de la isóbata, pero el modelo es menos fiable en celdas costeras.

In [ ]:
res.tabla[res.tabla["valido"] & ~res.tabla["cumple_criterios_basicos"]].sort_values(principal).head(10).round(2)

## 4. Salidas e insumos

In [ ]:
for nombre, ruta in res.salidas.items():
    print(f"{nombre:10s} {ruta}")
print()
print("Batimetría:", res.batimetria.origen.get("ruta"), res.batimetria.origen.get("sha256", "")[:16])
print("WAVERYS:   ", res.malla.origen.get("ruta"), res.malla.origen.get("sha256", "")[:16])

## Límites

- No se evalúan zonas de exclusión ni de colisión de usos, ni observaciones cercanas.
- Cerca de la costa la batimetría global interpola entre sondeos; WAVERYS usa su propia batimetría, más gruesa.
- El nodo representa una celda de 0.2° (unos 22 km): mirar la celda, no solo el punto.